<a href="https://colab.research.google.com/github/realnanayawjnr/demo-repo/blob/main/Parliament_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4

In [ ]:
import os
import requests
from bs4 import BeautifulSoup

# URL for Order Papers (change 'type=OP' to BS for Business Statements, etc.)
BASE_URL = "https://www.parliament.gh"
DOCS_URL = "https://www.parliament.gh/docs?type=OP"

# Folder to save PDFs
SAVE_DIR = "Parliament_PDFs"
os.makedirs(SAVE_DIR, exist_ok=True)

# Fetch the page
response = requests.get(DOCS_URL)
soup = BeautifulSoup(response.text, "html.parser")

# Find all PDF links
pdf_links = [BASE_URL + a["href"] for a in soup.find_all("a", href=True) if a["href"].endswith(".pdf")]

print(f"Found {len(pdf_links)} PDF files.")

# Download each PDF
for link in pdf_links:
    filename = link.split("/")[-1]
    filepath = os.path.join(SAVE_DIR, filename)
    if not os.path.exists(filepath):  # skip if already downloaded
        print(f"Downloading {filename}...")
        pdf_data = requests.get(link).content
        with open(filepath, "wb") as f:
            f.write(pdf_data)
print("✅ All PDFs downloaded!")


Found 0 PDF files.
✅ All PDFs downloaded!


In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import re

# Base URLs
BASE_URL = "https://www.parliament.gh"
DOCS_URL = "https://www.parliament.gh/docs?type=OP"

# Save folder
SAVE_DIR = "Parliament_PDFs"
os.makedirs(SAVE_DIR, exist_ok=True)

# Fetch page
response = requests.get(DOCS_URL)
soup = BeautifulSoup(response.text, "html.parser")

# Find onclick attributes with showPDF
pdf_links = []
for row in soup.find_all("tr", onclick=True):
    match = re.search(r"showPDF\('([^']+\.pdf)'", row["onclick"])
    if match:
        pdf_links.append(BASE_URL + "/" + match.group(1))

print(f"Found {len(pdf_links)} PDF files.")

# Download PDFs
for link in pdf_links:
    filename = link.split("/")[-1]
    filepath = os.path.join(SAVE_DIR, filename)
    if not os.path.exists(filepath):  # skip if already exists
        print(f"Downloading {filename}...")
        pdf_data = requests.get(link).content
        with open(filepath, "wb") as f:
            f.write(pdf_data)

print("✅ All PDFs downloaded!")


Found 50 PDF files.
✅ All PDFs downloaded!


In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import re

# Base URLs
BASE_URL = "https://www.parliament.gh"
DOCS_URL = "https://www.parliament.gh/docs?type=OP"

# Save folder
SAVE_DIR = "Parliament_PDFs"
os.makedirs(SAVE_DIR, exist_ok=True)

all_pdfs = []

# Loop through all 24 pages (0, 50, 100, …, 1150)
for skip in range(0, 1200, 50):
    url = f"{DOCS_URL}&skip={skip}"
    print(f"📄 Checking page: {url}")

    response = requests.get(url)
    if response.status_code != 200:
        print(f"⚠️ Failed to load page {url}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    # Extract PDF links from onclick attributes
    for row in soup.find_all("tr", onclick=True):
        match = re.search(r"showPDF\('([^']+\.pdf)'", row["onclick"])
        if match:
            pdf_url = BASE_URL + "/" + match.group(1)
            if pdf_url not in all_pdfs:
                all_pdfs.append(pdf_url)

print(f"\n✅ Found {len(all_pdfs)} unique PDF files across all pages.\n")

# Download PDFs
for link in all_pdfs:
    filename = link.split("/")[-1]
    filepath = os.path.join(SAVE_DIR, filename)
    if not os.path.exists(filepath):  # skip if already downloaded
        print(f"⬇️ Downloading {filename}...")
        pdf_data = requests.get(link).content
        with open(filepath, "wb") as f:
            f.write(pdf_data)

print("\n🎉 All PDFs downloaded successfully!")


📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=0
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=50
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=100
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=150
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=200
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=250
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=300
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=350
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=400
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=450
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=500
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=550
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=600
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=650
📄 Checking page: https://www.parliament.gh/docs?type=OP&skip=700
📄 Checking page: https://www

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import re

BASE_URL = "https://www.parliament.gh"
DOCS_BASE = "https://www.parliament.gh/docs"

# Sections to scrape
SECTIONS = {
    "AG": "Agenda",
    "BS": "Business_Statements",
    "OP": "Order_Papers",
    "VP": "Votes_Proceedings",
    "Acts&OT": "Acts",
    "Bills&OT": "Bills",
    "HS": "Official_Reports",
    # Publications will need extra handling (different structure, if PDFs exist)
    "factsheets": "Publications/Facts_Sheets",
    "policybriefs": "Publications/Policy_Briefs",
    "budgetbriefs": "Publications/Budget_Briefs",
    "backgroundpapers": "Publications/Background_Papers",
    "committeebriefs": "Publications/Committee_Briefs",
    "proceduralbriefs": "Publications/Procedural_Briefs",
    "costestimate": "Publications/Cost_Estimate_of_Bills"
}

# Max number of pages to crawl per section (24 seen for OP, adjust as needed)
MAX_PAGES = 30

def fetch_pdfs(section_code, folder_name):
    print(f"\n📂 Section: {folder_name}")
    os.makedirs(folder_name, exist_ok=True)

    for page_index, skip in enumerate(range(0, MAX_PAGES * 50, 50), start=1):
        url = f"{DOCS_BASE}?type={section_code}&skip={skip}"
        response = requests.get(url)
        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.text, "html.parser")

        # Find all PDFs from JS onclick attributes
        pdf_links = []
        for row in soup.find_all("tr", onclick=True):
            match = re.search(r"showPDF\('([^']+\.pdf)'", row["onclick"])
            if match:
                pdf_links.append(BASE_URL + "/" + match.group(1))

        if not pdf_links:
            break  # stop if no more PDFs

        print(f"   📄 Page {page_index}: Found {len(pdf_links)} PDFs")

        # Create subfolder for this page
        page_folder = os.path.join(folder_name, f"Page_{page_index}")
        os.makedirs(page_folder, exist_ok=True)

        # Download PDFs
        for link in pdf_links:
            filename = link.split("/")[-1]
            filepath = os.path.join(page_folder, filename)

            if not os.path.exists(filepath):
                print(f"     ⬇️ Downloading {filename}...")
                pdf_data = requests.get(link).content
                with open(filepath, "wb") as f:
                    f.write(pdf_data)

for code, folder in SECTIONS.items():
    fetch_pdfs(code, folder)

print("\n🎉 All sections downloaded successfully!")



📂 Section: Agenda
   📄 Page 1: Found 3 PDFs
     ⬇️ Downloading AGENDA_Second_Meeting_First_Session_May-July 2025..pdf...
     ⬇️ Downloading AGENDA-1st Mtg-Jan-Mar 2022.pdf...
     ⬇️ Downloading AGENDA-3rd Mtg-Oct 2018.pdf...
   📄 Page 2: Found 3 PDFs
     ⬇️ Downloading AGENDA_Second_Meeting_First_Session_May-July 2025..pdf...
     ⬇️ Downloading AGENDA-1st Mtg-Jan-Mar 2022.pdf...
     ⬇️ Downloading AGENDA-3rd Mtg-Oct 2018.pdf...
   📄 Page 3: Found 3 PDFs
     ⬇️ Downloading AGENDA_Second_Meeting_First_Session_May-July 2025..pdf...
     ⬇️ Downloading AGENDA-1st Mtg-Jan-Mar 2022.pdf...
     ⬇️ Downloading AGENDA-3rd Mtg-Oct 2018.pdf...
   📄 Page 4: Found 3 PDFs
     ⬇️ Downloading AGENDA_Second_Meeting_First_Session_May-July 2025..pdf...
     ⬇️ Downloading AGENDA-1st Mtg-Jan-Mar 2022.pdf...
     ⬇️ Downloading AGENDA-3rd Mtg-Oct 2018.pdf...
   📄 Page 5: Found 3 PDFs
     ⬇️ Downloading AGENDA_Second_Meeting_First_Session_May-July 2025..pdf...
     ⬇️ Downloading AGENDA-1st Mtg-Ja